# Assignment 01: Linear Algebra for Deep Learning

**Module:** 02 — Math for Deep Learning  
**Estimated Time:** 6-8 hours  
**Language:** Python (NumPy, PyTorch for verification)

---

## Learning Objectives

- Implement matrix operations from scratch to build intuition for neural network forward passes
- Visualize 2D linear transformations and connect them to what neural network layers do
- Implement PCA from eigendecomposition and apply it to MNIST
- Understand SVD-based image compression and its connection to low-rank approximations
- Master NumPy broadcasting rules

### Key Equations

**Matrix-vector multiplication** (linear layer forward pass):
$$\mathbf{y} = \mathbf{W}\mathbf{x} + \mathbf{b}, \quad y_i = \sum_j W_{ij} x_j + b_i$$

**Covariance matrix:**
$$\mathbf{C} = \frac{1}{n-1} (\mathbf{X} - \bar{\mathbf{X}})^T (\mathbf{X} - \bar{\mathbf{X}})$$

**SVD:**
$$\mathbf{A} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^T, \quad \mathbf{A}_k = \sum_{i=1}^{k} \sigma_i \mathbf{u}_i \mathbf{v}_i^T$$

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA as SklearnPCA

sys.path.insert(0, '../../')
from shared_utils.common import set_seed

set_seed(42)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('Setup complete.')

---
## Exercise 1: Matrix Operations from Scratch

**Why this matters:** Every neural network forward pass is a sequence of matrix operations. Implement using only basic NumPy (no `np.linalg`, no `np.matmul` for core implementation).

In [ ]:
def dot_product(a: np.ndarray, b: np.ndarray) -> float:
    """Compute the dot product of two 1D arrays.
    Do NOT use np.dot, np.inner, or the @ operator.
    
    Complexity: O(n)

    Args:
        a: 1D numpy array of shape (n,)
        b: 1D numpy array of shape (n,)
    Returns:
        Scalar dot product
    """
    # YOUR CODE HERE
    pass


def matvec(A: np.ndarray, x: np.ndarray) -> np.ndarray:
    """Compute A @ x using only dot products.
    Each output element = dot product of one ROW of A and x.
    This is exactly what a single linear layer does.
    
    Complexity: O(m * n)

    Args:
        A: 2D array of shape (m, n)
        x: 1D array of shape (n,)
    Returns:
        1D array of shape (m,)
    """
    # YOUR CODE HERE
    pass


def matmul(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """Compute A @ B using only matvec.
    
    Complexity: O(m * k * n)

    Args:
        A: 2D array of shape (m, k)
        B: 2D array of shape (k, n)
    Returns:
        2D array of shape (m, n)
    """
    # YOUR CODE HERE
    pass


def outer_product(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Compute the outer product a @ b^T.
    This is how weight gradients are computed: dL/dW = dL/dz outer input.
    
    Complexity: O(m * n)

    Args:
        a: 1D array of shape (m,)
        b: 1D array of shape (n,)
    Returns:
        2D array of shape (m, n)
    """
    # YOUR CODE HERE
    pass


def transpose(A: np.ndarray) -> np.ndarray:
    """Compute the transpose without using .T or np.transpose.
    
    Complexity: O(m * n)

    Args:
        A: 2D array of shape (m, n)
    Returns:
        2D array of shape (n, m)
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Test Exercise 1 ---
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
assert np.isclose(dot_product(a, b), np.dot(a, b)), 'dot_product failed'

A = np.random.randn(3, 4)
x = np.random.randn(4)
assert np.allclose(matvec(A, x), A @ x), 'matvec failed'

B = np.random.randn(4, 5)
assert np.allclose(matmul(A, B), A @ B), 'matmul failed'

# Test with various sizes
for shape in [(10, 20, 3), (1, 5, 1), (7, 7, 7)]:
    m, k, n = shape
    A2 = np.random.randn(m, k)
    B2 = np.random.randn(k, n)
    assert np.allclose(matmul(A2, B2), A2 @ B2), f'matmul failed for shapes ({m},{k}) @ ({k},{n})'

assert np.allclose(outer_product(a, b), np.outer(a, b)), 'outer_product failed'
assert np.allclose(transpose(A), A.T), 'transpose failed'

print('Exercise 1 all tests passed!')

**How does matrix multiplication cost scale? Why does this matter for neural network width?**

*YOUR ANSWER HERE (3-4 sentences)*

---
## Exercise 2: Visualizing 2D Linear Transformations

**Why this matters:** The weight matrix of a linear layer transforms input space into output space. Visualize what different matrices "do" to understand what layers learn.

In [ ]:
def make_F():
    """Create a set of 2D points forming the letter F."""
    points = []
    for y in np.linspace(0, 2, 20):
        points.append([0, y])
    for x in np.linspace(0, 1.5, 15):
        points.append([x, 2])
    for x in np.linspace(0, 1.0, 10):
        points.append([x, 1])
    return np.array(points)


def plot_transformation(original, transformed, title, matrix_str, det, ax):
    """Plot original points in blue and transformed in red."""
    # YOUR CODE HERE
    # 1. Plot original points in blue
    # 2. Plot transformed points in red
    # 3. Draw arrows for a few selected points
    # 4. Add origin and axis lines
    # 5. Title with matrix info and determinant
    pass

In [ ]:
# --- Exercise 2b: Apply 6 transformations ---
F = make_F()

transformations = {
    'Rotation 45deg': np.array([[np.cos(np.pi/4), -np.sin(np.pi/4)],
                                [np.sin(np.pi/4),  np.cos(np.pi/4)]]),
    'Non-uniform Scale': np.array([[2.0, 0.0], [0.0, 0.5]]),
    'Horizontal Shear': np.array([[1.0, 0.5], [0.0, 1.0]]),
    'Reflection (x-axis)': np.array([[1.0, 0.0], [0.0, -1.0]]),
    'Projection (x-axis)': np.array([[1.0, 0.0], [0.0, 0.0]]),
    'Random NN Layer': np.random.RandomState(42).randn(2, 2) * 0.5
}

# YOUR CODE HERE
# Create a 2x3 grid of subplots showing each transformation
# For each, compute: transformed = (matrix @ F.T).T
# Label each with the matrix, determinant, and what it does
pass

In [ ]:
# --- Exercise 2c: Composition of transformations ---
# YOUR CODE HERE
# 1. Rotation(30) then Scale(2, 0.5): apply R first, then S -> S @ R
# 2. Scale(2, 0.5) then Rotation(30): apply S first, then R -> R @ S
# 3. Verify sequential application == single matrix
# 4. Show S @ R != R @ S
pass

**Why is the bias necessary? What can affine transformations do that pure linear transformations cannot?**

*YOUR ANSWER HERE (4-6 sentences)*

---
## Exercise 3: PCA from Scratch

**Why this matters:** PCA finds the axes of maximum variance using eigendecomposition. Understanding PCA means understanding eigenvalues, covariance matrices, and projection.

In [ ]:
class PCA:
    """Principal Component Analysis from scratch."""

    def __init__(self, n_components: int):
        self.n_components = n_components
        self.components: np.ndarray = None   # (n_components, n_features)
        self.mean: np.ndarray = None          # (n_features,)
        self.explained_variance: np.ndarray = None  # eigenvalues

    def fit(self, X: np.ndarray) -> 'PCA':
        """Fit PCA on data X.

        Steps:
        1. Center the data (subtract mean)
        2. Compute covariance matrix
        3. Compute eigenvalues and eigenvectors
        4. Sort by eigenvalue (descending)
        5. Store top n_components eigenvectors

        Args:
            X: shape (n_samples, n_features)
        """
        # YOUR CODE HERE
        pass

    def transform(self, X: np.ndarray) -> np.ndarray:
        """Project data onto principal components.

        Args:
            X: shape (n_samples, n_features)
        Returns:
            shape (n_samples, n_components)
        """
        # YOUR CODE HERE
        pass

    def inverse_transform(self, X_projected: np.ndarray) -> np.ndarray:
        """Reconstruct data from projected representation.

        Args:
            X_projected: shape (n_samples, n_components)
        Returns:
            shape (n_samples, n_features)
        """
        # YOUR CODE HERE
        pass

    def explained_variance_ratio(self) -> np.ndarray:
        """Return proportion of variance explained by each component."""
        # YOUR CODE HERE
        pass

In [ ]:
# --- Test Exercise 3: Verify against sklearn ---
np.random.seed(42)
X_pca = np.random.multivariate_normal(
    mean=[2, 3], cov=[[3, 2], [2, 2]], size=200
)

my_pca = PCA(n_components=2)
my_pca.fit(X_pca)

sk_pca = SklearnPCA(n_components=2)
sk_pca.fit(X_pca)

print('My PCA variance ratio:', my_pca.explained_variance_ratio())
print('sklearn variance ratio:', sk_pca.explained_variance_ratio_)

# Ratios should match (components may differ in sign)
assert np.allclose(
    np.abs(my_pca.explained_variance_ratio()),
    np.abs(sk_pca.explained_variance_ratio_),
    atol=0.01
), 'Explained variance ratios should match'
print('Exercise 3 verification passed!')

In [ ]:
# --- Exercise 3c: Visualize PCA on 2D data ---
# YOUR CODE HERE
# 1. Plot original data
# 2. Overlay principal component directions as arrows (scaled by eigenvalues)
# 3. Show reconstruction from first component only
pass

---
## Exercise 4: PCA on MNIST

Apply PCA to MNIST (784-dimensional images) and analyze compression, reconstruction, and 2D visualization.

In [ ]:
# --- Exercise 4a: Load MNIST ---
from sklearn.datasets import fetch_openml

print('Loading MNIST (this may take a minute)...')
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X_mnist = mnist.data.astype(np.float64)
y_mnist = mnist.target.astype(int)

# Use a subset for speed
X_subset = X_mnist[:10000]
y_subset = y_mnist[:10000]
print(f'Loaded: {X_subset.shape}')

In [ ]:
# --- Exercise 4b: Cumulative variance ---
# YOUR CODE HERE
# 1. Fit PCA with n_components=50 on the subset
# 2. Plot cumulative explained variance ratio
# 3. Mark 90%, 95%, 99% thresholds
# 4. Print how many components needed for each threshold
pass

In [ ]:
# --- Exercise 4c: Principal components as images ---
# YOUR CODE HERE
# Display top 10 principal components as 28x28 images
# Use cmap='RdBu_r'
pass

**What do the principal components look like? Do they resemble digits?**

*YOUR ANSWER HERE*

In [ ]:
# --- Exercise 4d: Reconstruction at different compression levels ---
# YOUR CODE HERE
# For k in [5, 10, 20, 50, 100, 200], show reconstruction of sample digits
pass

**At what number of components are reconstructions "good enough"?**

*YOUR ANSWER HERE*

In [ ]:
# --- Exercise 4e: 2D scatter plot ---
# YOUR CODE HERE
# Project to 2D, scatter plot colored by digit label
pass

**Which digits are well-separated? Which overlap? Why?**

*YOUR ANSWER HERE*

---
## Exercise 5: SVD-Based Image Compression

**Why this matters:** SVD gives the best low-rank approximation. This is the foundation for LoRA used in fine-tuning large language models.

In [ ]:
def svd_compress(U: np.ndarray, s: np.ndarray, Vt: np.ndarray, k: int) -> np.ndarray:
    """Reconstruct image using top-k singular values."""
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]


def compression_ratio(original_shape: tuple, k: int) -> float:
    """Compute the compression ratio for rank-k approximation."""
    m, n = original_shape
    original_size = m * n
    compressed_size = k * (m + 1 + n)
    return original_size / compressed_size

In [ ]:
# --- Exercise 5a-b: SVD compression ---
# Generate a test image (or load a real one)
np.random.seed(42)
# Create a structured image (gradient pattern)
x_coords = np.linspace(0, 1, 256)
y_coords = np.linspace(0, 1, 256)
X_img, Y_img = np.meshgrid(x_coords, y_coords)
img = np.sin(5 * X_img) * np.cos(3 * Y_img) + 0.5 * X_img

U, s, Vt = np.linalg.svd(img, full_matrices=False)

# YOUR CODE HERE
# Show reconstructions for ranks [1, 5, 10, 20, 50, 100]
# Include compression ratio and reconstruction error for each
# Plot singular value spectrum in the last subplot
pass

In [ ]:
# --- Exercise 5c: Cumulative energy ---
# YOUR CODE HERE
# Plot singular values (log scale) and cumulative energy
# Find rank for 90%, 95%, 99% energy
pass

**What does rapidly decaying singular values mean about image dimensionality?**

*YOUR ANSWER HERE*

In [ ]:
# --- Exercise 5d: Compare image types ---
# YOUR CODE HERE
# 1. Natural image (smooth pattern, already done above)
# 2. Simple geometric pattern (e.g., stripes)
# 3. Pure noise: np.random.randn(256, 256)
# Compare singular value spectra and reconstruction quality
pass

**Which image type compresses best? Worst? Why?**

*YOUR ANSWER HERE*

---
## Exercise 6: Broadcasting — Predict Then Verify

For each scenario, first **predict** the output shape, then run the code to verify.

In [ ]:
# Scenario 1
# Prediction: shape = ?, succeeds = ?
A = np.ones((3, 4))
b = np.ones((4,))
# YOUR PREDICTION HERE, then uncomment:
# result = A + b
# print(f'Scenario 1: shape = {result.shape}')

In [ ]:
# Scenario 2
# Prediction: shape = ?, succeeds = ?
A = np.ones((3, 4))
b = np.ones((3,))
# YOUR PREDICTION AND VERIFICATION HERE

In [ ]:
# Scenario 3
# Prediction: shape = ?, succeeds = ?
A = np.ones((3, 4))
b = np.ones((3, 1))
# YOUR PREDICTION AND VERIFICATION HERE

In [ ]:
# Scenario 4
# Prediction: shape = ?, succeeds = ?
A = np.ones((2, 3, 4))
b = np.ones((3, 4))
# YOUR PREDICTION AND VERIFICATION HERE

In [ ]:
# Scenario 5
# Prediction: shape = ?, succeeds = ?
A = np.ones((2, 3, 4))
b = np.ones((2, 1, 4))
# YOUR PREDICTION AND VERIFICATION HERE

In [ ]:
# Scenario 6: The Trap
# Prediction: shape = ?, succeeds = ?
logits = np.random.randn(32, 10)
labels = np.arange(32)  # shape (32,)
# result = logits - labels
# What shape is this? Is it what you want?
# YOUR PREDICTION AND EXPLANATION HERE

In [ ]:
# Scenario 7: Batch normalization pattern
# Prediction for each step:
activations = np.random.randn(32, 256)
mean = activations.mean(axis=0)        # shape = ?
std = activations.std(axis=0)          # shape = ?
# normalized = (activations - mean) / std  # broadcasts correctly?
# YOUR PREDICTION AND VERIFICATION HERE

In [ ]:
# Scenario 8: Attention mask pattern
# Prediction: shape = ?, succeeds = ?
scores = np.random.randn(8, 16, 64, 64)
mask = np.ones((1, 1, 64, 64))
# masked_scores = scores + mask
# YOUR PREDICTION AND VERIFICATION HERE

**For any wrong predictions, explain what you misunderstood:**

*YOUR NOTES HERE*